# SD3.5 BDD100K Road-Scene Img2Img Augmentation Pipeline

Single-file Kaggle notebook for BDD100K road-facing dashcam augmentation. It scans the Kaggle BDD100K `images/100k/{train,val,test}` folders, records optional labels, and generates batched image-to-image variants for road-scene data augmentation.

## 1. Install Dependencies

Keep Kaggle's preinstalled CUDA/PyTorch stack intact. Do not reinstall `torch`, `torchvision`, or `xformers`, because that can trigger CUDA, RAPIDS, `numba`, and `cuda-core` dependency conflicts.

In [ ]:
!pip install -q \
  "diffusers>=0.30.0,<1.0.0" \
  "transformers>=4.40.0" \
  "peft>=0.11.0" \
  "accelerate>=0.30.0" \
  "bitsandbytes>=0.43.0" \
  sentencepiece protobuf safetensors

## 2. Runtime Check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda runtime:", torch.version.cuda)
    print("gpu count:", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(f"gpu {index}:", torch.cuda.get_device_name(index))

## 3. Hugging Face Login

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face with Kaggle secret HF_TOKEN.")
except Exception as exc:
    print("HF login skipped or failed. Add Kaggle secret HF_TOKEN if the model is gated.")
    print(type(exc).__name__, exc)

## 4. Configuration

Mount your Kaggle dataset and set `DATASET_ROOT` to the folder containing the `bdd100k/images/100k` and optional `bdd100k/labels` folders. The notebook waits cleanly if the folder is not mounted yet.

In [ ]:
from pathlib import Path

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/alvaromalfaro/bdd100k/bdd100k"),
    Path("/kaggle/input/bdd100k/bdd100k"),
    Path("/kaggle/input/BDD100K/bdd100k"),
    Path("/kaggle/input/solesensei-bdd100k/bdd100k"),
    Path("/kaggle/input/solesensei_bdd100k/bdd100k"),
]
DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), DATASET_ROOT_CANDIDATES[0])
IMAGE_ROOT = DATASET_ROOT / "images" / "100k"
LABEL_ROOT = DATASET_ROOT / "labels"
DATASET_SPLIT_DIRS = {
    "train": IMAGE_ROOT / "train",
    "val": IMAGE_ROOT / "val",
    "test": IMAGE_ROOT / "test",
}
LABEL_SPLIT_DIRS = {
    "train": LABEL_ROOT / "train",
    "val": LABEL_ROOT / "val",
    "test": LABEL_ROOT / "test",
}
SCENE_BUCKETS = ["road_scene"]
IMAGE_SUBDIR = "images"
CAPTION_CSV = None  # Optional CSV with columns: file_name, caption

MODEL_BACKEND = "sd35"  # "sd35" target or "sdxl" baseline
SD35_MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
SDXL_MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

OUTPUT_DIR = Path("/kaggle/working/generated_bdd100k_road_aug")
LORA_DIR = Path("/kaggle/working/sd35-lora-output")

RESOLUTION = 512
LORA_RANK = 16  # use 8 if OOM
TRAIN_BATCH_SIZE = 1
MAX_TRAIN_IMAGES = 500  # fast smoke-test scan; set None only when you really need all images
MAX_TRAIN_STEPS = 300
LEARNING_RATE = 1e-4
USE_T5 = False  # False is safer on Kaggle T4
TRAIN_DEVICE = "cuda:0"
USE_ALL_GPUS_FOR_AUGMENTATION = True
AUGMENTATION_DEVICES = None  # None = auto-detect all CUDA GPUs, e.g. ["cuda:0", "cuda:1"] on Kaggle T4 x2
MATCH_LABEL_JSON = True  # Reads BDD100K weather/timeofday/scene labels for weather-conditioned LoRA training

AUGMENTATION_VARIANTS = ["clear_day", "dusk_low_light", "rainy_road", "foggy_road", "motion_blur", "exposure_shift"]
AUGMENTATIONS_PER_BUCKET = 200
TARGET_SPLITS = ["train", "val", "test"]
AUGMENTATION_STRENGTH = 0.38
GUIDANCE_SCALE = 6.0
NUM_INFERENCE_STEPS = 32
VARIANT_STRENGTHS = {
    "clear_day": 0.42,
    "dusk_low_light": 0.52,
    "rainy_road": 0.56,
    "foggy_road": 0.54,
    "motion_blur": 0.34,
    "exposure_shift": 0.30,
}
VARIANT_GUIDANCE_SCALES = {
    "clear_day": 6.5,
    "dusk_low_light": 7.5,
    "rainy_road": 7.5,
    "foggy_road": 7.0,
    "motion_blur": 5.5,
    "exposure_shift": 5.0,
}
SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR.mkdir(parents=True, exist_ok=True)

## 5. Imports And Prompt Templates

In [ ]:
import csv
import gc
import json
import math
import os
import random
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from threading import Lock
from typing import Optional

os.environ["DIFFUSERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="Flax classes are deprecated.*")
warnings.filterwarnings("ignore", category=FutureWarning, module="diffusers.*")

import bitsandbytes as bnb
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from diffusers import (
    StableDiffusion3Img2ImgPipeline,
    StableDiffusion3Pipeline,
    StableDiffusionXLImg2ImgPipeline,
)
from diffusers.utils import logging as diffusers_logging
from diffusers.models.attention_processor import AttnProcessor2_0
from peft import LoraConfig, get_peft_model
from PIL import Image, ImageOps

diffusers_logging.set_verbosity_error()

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
BASE_CAPTION = "BDD100K dashcam road scene, preserve road layout and traffic objects"

SCENE_PROMPTS = {
    "road_scene": "realistic driving scene",
}

VARIANT_PROMPTS = {
    "clear_day": "convert to clear sunny daytime, dry road, blue sky",
    "dusk_low_light": "convert to dusk low light, headlights on, street lights",
    "rainy_road": "convert to rainy weather, wet windshield, wet asphalt, reflections",
    "foggy_road": "convert to foggy weather, dense haze, low visibility",
    "motion_blur": "subtle dashcam motion blur",
    "exposure_shift": "camera exposure shift, realistic tone",
}

VARIANT_TARGET_WEATHER = {
    "clear_day": "clear",
    "rainy_road": "rainy",
    "foggy_road": "foggy",
}

NEGATIVE_PROMPT = (
    "changed road layout, changed lane geometry, removed vehicles, added vehicles, missing pedestrians, "
    "warped traffic signs, distorted perspective, text, watermark, cartoon, synthetic artifacts, severe blur"
)

PIPELINE_LOAD_LOCK = Lock()

## 6. Dataset Scanner

This scans BDD100K image splits from `images/100k/{train,val,test}`. Optional label JSON files under `labels/{train,val,test}` are recorded when a matching file is present.

In [ ]:
@dataclass
class ImageRecord:
    path: Path
    split: str
    bucket: str
    caption: str
    label_path: Optional[Path] = None
    weather: Optional[str] = None
    timeofday: Optional[str] = None
    scene: Optional[str] = None


def load_caption_map(csv_path):
    if not csv_path:
        return {}
    csv_path = Path(csv_path)
    if not csv_path.exists():
        print(f"Caption CSV not found: {csv_path}. Using default captions.")
        return {}

    caption_map = {}
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            file_name = row.get("file_name") or row.get("filename") or row.get("image")
            caption = row.get("caption") or row.get("prompt")
            if file_name and caption:
                caption_map[Path(file_name).name] = caption
    return caption_map


def find_label_path(image_path, label_dir):
    label_dir = Path(label_dir)
    if not label_dir.exists():
        return None
    candidates = [
        label_dir / f"{image_path.stem}.json",
        label_dir / f"{image_path.name}.json",
        label_dir / image_path.stem / "label.json",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def find_split_label_file(label_dir):
    label_dir = Path(label_dir)
    if not label_dir.exists():
        return None
    first_json = None
    json_count = 0
    with os.scandir(label_dir) as entries:
        for entry in entries:
            if not entry.is_file() or not entry.name.lower().endswith(".json"):
                continue
            path = Path(entry.path)
            json_count += 1
            if first_json is None:
                first_json = path
            if "label" in entry.name.lower() or "bdd100k" in entry.name.lower():
                return path
            if json_count > 1:
                return None
    return first_json if json_count == 1 else None


def clean_label_value(value):
    if value is None:
        return None
    value = str(value).strip().lower().replace("_", " ")
    if not value or value in {"none", "undefined", "unknown"}:
        return None
    return value


def extract_bdd_attributes(label_data):
    if not isinstance(label_data, dict):
        return {}
    attrs = label_data.get("attributes") or label_data.get("attr") or {}
    if not isinstance(attrs, dict):
        attrs = {}
    return {
        "weather": clean_label_value(attrs.get("weather") or label_data.get("weather")),
        "timeofday": clean_label_value(attrs.get("timeofday") or attrs.get("time_of_day") or label_data.get("timeofday")),
        "scene": clean_label_value(attrs.get("scene") or label_data.get("scene")),
    }


def load_split_label_index(label_dir):
    if not MATCH_LABEL_JSON:
        return {}, None
    label_dir = Path(label_dir)
    split_label_file = find_split_label_file(label_dir)
    if split_label_file:
        try:
            with split_label_file.open("r", encoding="utf-8") as handle:
                data = json.load(handle)
            if isinstance(data, dict):
                for key in ("frames", "images", "annotations", "items"):
                    if isinstance(data.get(key), list):
                        data = data[key]
                        break
            if isinstance(data, list):
                index = {}
                for item in data:
                    if not isinstance(item, dict):
                        continue
                    name = item.get("name") or item.get("file_name") or item.get("filename") or item.get("image")
                    if name:
                        index[Path(name).name] = extract_bdd_attributes(item)
                return index, split_label_file
        except Exception as exc:
            print(f"Could not read split label file {split_label_file}: {type(exc).__name__}: {exc}")
    return {}, split_label_file


def read_per_image_attributes(image_path, label_dir):
    label_path = find_label_path(image_path, label_dir)
    if not label_path:
        return {}, None
    try:
        with label_path.open("r", encoding="utf-8") as handle:
            return extract_bdd_attributes(json.load(handle)), label_path
    except Exception:
        return {}, label_path


def metadata_phrase(metadata, include_weather=True):
    parts = []
    if include_weather and metadata.get("weather"):
        parts.append(f"weather {metadata['weather']}")
    if metadata.get("timeofday"):
        parts.append(f"time {metadata['timeofday']}")
    if metadata.get("scene"):
        parts.append(f"scene {metadata['scene']}")
    return ", ".join(parts)


def build_caption(path, bucket, caption_map, metadata=None, include_weather=True):
    if path.name in caption_map:
        return caption_map[path.name]
    metadata = metadata or {}
    pieces = [BASE_CAPTION, SCENE_PROMPTS[bucket]]
    label_text = metadata_phrase(metadata, include_weather=include_weather)
    if label_text:
        pieces.append(label_text)
    return ", ".join(pieces)


def build_generation_prompt(record, variant):
    metadata = {"timeofday": record.timeofday, "scene": record.scene}
    base = build_caption(record.path, record.bucket, {}, metadata=metadata, include_weather=False)
    return f"{VARIANT_PROMPTS[variant]}, {base}"


def build_variant_negative_prompt(variant):
    variant_negatives = {
        "clear_day": "rain, fog, night, wet windshield, wet road",
        "dusk_low_light": "bright noon, overexposed sunlight",
        "rainy_road": "dry road, clear sky, sunny weather",
        "foggy_road": "clear visibility, sharp distant background",
    }
    extra = variant_negatives.get(variant, "")
    return f"{NEGATIVE_PROMPT}, {extra}" if extra else NEGATIVE_PROMPT


def list_image_paths_fast(split_dir, max_images=None):
    split_dir = Path(split_dir)
    image_paths = []
    with os.scandir(split_dir) as entries:
        for entry in entries:
            if not entry.is_file():
                continue
            path = Path(entry.path)
            if path.suffix.lower() not in IMAGE_EXTS:
                continue
            image_paths.append(path)
            if max_images and len(image_paths) >= max_images:
                break
    return image_paths


def scan_dataset(split_dirs=DATASET_SPLIT_DIRS, caption_csv=CAPTION_CSV, max_images=MAX_TRAIN_IMAGES):
    missing = [str(path) for path in split_dirs.values() if not Path(path).exists()]
    if missing:
        print("WAITING FOR DATASET. Missing split folders:")
        for path in missing:
            print(" -", path)
        print("Mount the solesensei_bdd100k Kaggle dataset, then update DATASET_ROOT/DATASET_SPLIT_DIRS and rerun this cell.")
        return []

    caption_map = load_caption_map(caption_csv)
    records = []
    for split, split_dir in split_dirs.items():
        split_dir = Path(split_dir)
        bucket = SCENE_BUCKETS[0]
        label_dir = LABEL_SPLIT_DIRS.get(split, LABEL_ROOT / split)
        label_index, split_label_path = load_split_label_index(label_dir)
        image_paths = list_image_paths_fast(split_dir, max_images=max_images)
        for path in image_paths:
            metadata = label_index.get(path.name, {})
            per_image_label_path = None
            if MATCH_LABEL_JSON and not metadata:
                metadata, per_image_label_path = read_per_image_attributes(path, label_dir)
            label_path = per_image_label_path or (split_label_path if metadata else None)
            records.append(ImageRecord(
                path=path,
                split=split,
                bucket=bucket,
                caption=build_caption(path, bucket, caption_map, metadata=metadata, include_weather=True),
                label_path=label_path,
                weather=metadata.get("weather"),
                timeofday=metadata.get("timeofday"),
                scene=metadata.get("scene"),
            ))

    print(f"Found {len(records)} BDD100K road-scene images")
    for split in split_dirs:
        count = sum(1 for record in records if record.split == split)
        labels = sum(1 for record in records if record.split == split and record.label_path)
        weather = sum(1 for record in records if record.split == split and record.weather)
        print(f"{split:5s} | images={count:6d} | matched_label_json={labels:6d} | weather_labels={weather:6d}")
    if max_images:
        print(f"Fast scan mode: capped at {max_images} images per split. Set MAX_TRAIN_IMAGES=None for a full scan.")
    return records


records = scan_dataset()
records[:3]

In [ ]:
def summarize_bdd_attributes(records):
    for attr in ["weather", "timeofday", "scene"]:
        counts = {}
        for record in records:
            value = getattr(record, attr) or "missing"
            counts[value] = counts.get(value, 0) + 1
        print(f"\n{attr} distribution:")
        for value, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
            print(f"  {value:16s} {count:6d}")


summarize_bdd_attributes(records)


In [ ]:
def preview_prompt_samples(records, variants=("rainy_road", "foggy_road", "dusk_low_light"), n=3):
    for record in records[:n]:
        print("\nimage:", record.path.name)
        print("label:", {"weather": record.weather, "timeofday": record.timeofday, "scene": record.scene})
        print("train caption:", record.caption)
        for variant in variants:
            print(f"{variant} prompt:", build_generation_prompt(record, variant))


preview_prompt_samples(records)


## 7. Dataset Preview

In [ ]:
def preview_records(records, n=6):
    if not records:
        print("No dataset records to preview yet.")
        return
    sample = records[:n]
    cols = min(3, len(sample))
    rows = math.ceil(len(sample) / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, record in enumerate(sample, 1):
        image = Image.open(record.path).convert("RGB")
        plt.subplot(rows, cols, i)
        plt.imshow(image)
        plt.title(f"{record.split}/{record.bucket}\n{record.path.name[:32]}")
        plt.axis("off")
    plt.tight_layout()


preview_records(records)

## 8. Image Preprocessing

In [ ]:
def load_source_image(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB")


def resize_center_crop(image, resolution=RESOLUTION):
    width, height = image.size
    scale = resolution / min(width, height)
    new_size = (round(width * scale), round(height * scale))
    image = image.resize(new_size, Image.BICUBIC)
    left = (image.width - resolution) // 2
    top = (image.height - resolution) // 2
    return image.crop((left, top, left + resolution, top + resolution))


def image_to_tensor(image, resolution=RESOLUTION, device="cuda", dtype=torch.float16):
    image = resize_center_crop(image, resolution)
    pixel_values = torch.tensor(list(image.getdata()), dtype=torch.float32).view(resolution, resolution, 3)
    pixel_values = pixel_values.permute(2, 0, 1).unsqueeze(0) / 127.5 - 1.0
    return pixel_values.to(device=device, dtype=dtype)


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. SD3.5 LoRA Training Across Dataset

This trains LoRA over multiple images from `records`. Keep `TRAIN_BATCH_SIZE=1` on T4. Start with small `MAX_TRAIN_STEPS` to validate memory.

In [ ]:
def get_sigmas(noise_scheduler, timesteps, n_dim, dtype, device):
    sigmas = noise_scheduler.sigmas.to(device=device, dtype=dtype)
    schedule_timesteps = noise_scheduler.timesteps.to(device)
    step_indices = []
    for timestep in timesteps:
        matches = (schedule_timesteps == timestep).nonzero()
        step_indices.append(matches[0].item() if len(matches) else 0)
    sigma = sigmas[step_indices].flatten()
    while len(sigma.shape) < n_dim:
        sigma = sigma.unsqueeze(-1)
    return sigma


def load_sd35_pipeline_for_training(model_id=SD35_MODEL_ID, use_t5=USE_T5):
    kwargs = {
        "torch_dtype": torch.float16,
        "use_safetensors": True,
    }
    if not use_t5:
        kwargs.update({"text_encoder_3": None, "tokenizer_3": None})
    try:
        return StableDiffusion3Pipeline.from_pretrained(model_id, **kwargs)
    except TypeError:
        kwargs.pop("text_encoder_3", None)
        kwargs.pop("tokenizer_3", None)
        return StableDiffusion3Pipeline.from_pretrained(model_id, **kwargs)


def train_sd35_lora_on_records(records, max_steps=MAX_TRAIN_STEPS, rank=LORA_RANK):
    if not records:
        raise FileNotFoundError("No images found. Mount dataset folder and rerun scan_dataset().")
    if TRAIN_BATCH_SIZE != 1:
        raise ValueError("Kaggle T4 path is configured for TRAIN_BATCH_SIZE=1.")

    device = TRAIN_DEVICE
    pipe = load_sd35_pipeline_for_training()
    pipe.scheduler.set_timesteps(1000, device=device)

    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    pipe.text_encoder_2.requires_grad_(False)
    if getattr(pipe, "text_encoder_3", None) is not None:
        pipe.text_encoder_3.requires_grad_(False)

    pipe.vae.to(device)
    pipe.transformer.to(device)
    pipe.transformer.set_attn_processor(AttnProcessor2_0())
    pipe.transformer.enable_gradient_checkpointing()

    lora_config = LoraConfig(
        r=rank,
        lora_alpha=rank,
        target_modules=["to_q", "to_k", "to_v", "to_out.0"],
        init_lora_weights="gaussian",
    )
    pipe.transformer = get_peft_model(pipe.transformer, lora_config)
    pipe.transformer.print_trainable_parameters()
    pipe.transformer.train()

    optimizer = bnb.optim.AdamW8bit(pipe.transformer.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    timesteps_schedule = pipe.scheduler.timesteps.to(device)

    for step in range(max_steps):
        record = records[step % len(records)]
        image = load_source_image(record.path)
        pixel_values = image_to_tensor(image, resolution=RESOLUTION, device=device, dtype=torch.float16)

        with torch.no_grad():
            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
            scaling = getattr(pipe.vae.config, "scaling_factor", 1.0)
            shift = getattr(pipe.vae.config, "shift_factor", 0.0)
            latents = (latents - shift) * scaling

            prompt_embeds, _, pooled_prompt_embeds, _ = pipe.encode_prompt(
                prompt=record.caption,
                prompt_2=record.caption,
                prompt_3=record.caption if USE_T5 else None,
                device=device,
                num_images_per_prompt=1,
                do_classifier_free_guidance=False,
            )

        noise = torch.randn_like(latents)
        timestep_index = torch.randint(0, len(timesteps_schedule), (1,), device=device)
        timesteps = timesteps_schedule[timestep_index]
        sigmas = get_sigmas(pipe.scheduler, timesteps, latents.ndim, latents.dtype, device)
        noisy_latents = sigmas * noise + (1.0 - sigmas) * latents
        target = noise - latents

        model_pred = pipe.transformer(
            hidden_states=noisy_latents,
            timestep=timesteps,
            encoder_hidden_states=prompt_embeds,
            pooled_projections=pooled_prompt_embeds,
            return_dict=False,
        )[0]

        loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        if step == 0 or (step + 1) % 25 == 0:
            print(f"step {step + 1}/{max_steps} | loss={loss.item():.5f} | image={record.path.name}")

    LORA_DIR.mkdir(parents=True, exist_ok=True)
    pipe.transformer.save_pretrained(LORA_DIR / "transformer_peft")
    print(f"Saved PEFT transformer LoRA to {LORA_DIR / 'transformer_peft'}")
    return pipe


# Run when dataset is mounted:
# trained_pipe = train_sd35_lora_on_records(records, max_steps=MAX_TRAIN_STEPS, rank=LORA_RANK)

## 10. Img2Img Augmentation Pipeline

In [ ]:
def resolve_augmentation_devices():
    if AUGMENTATION_DEVICES:
        return AUGMENTATION_DEVICES
    if not torch.cuda.is_available():
        return ["cpu"]
    if USE_ALL_GPUS_FOR_AUGMENTATION:
        return [f"cuda:{index}" for index in range(torch.cuda.device_count())]
    return [TRAIN_DEVICE]


def build_img2img_pipeline(backend=MODEL_BACKEND, lora_dir=None, device=TRAIN_DEVICE):
    if backend == "sd35":
        pipeline_cls = StableDiffusion3Img2ImgPipeline
        model_id = SD35_MODEL_ID
        kwargs = {"torch_dtype": torch.float16, "use_safetensors": True, "low_cpu_mem_usage": False}
        if not USE_T5:
            kwargs.update({"text_encoder_3": None, "tokenizer_3": None})
    elif backend == "sdxl":
        pipeline_cls = StableDiffusionXLImg2ImgPipeline
        model_id = SDXL_MODEL_ID
        kwargs = {"torch_dtype": torch.float16, "use_safetensors": True, "low_cpu_mem_usage": False}
    else:
        raise ValueError(f"Unsupported backend: {backend}")

    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)

    with PIPELINE_LOAD_LOCK:
        try:
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)
        except TypeError:
            kwargs.pop("text_encoder_3", None)
            kwargs.pop("tokenizer_3", None)
            kwargs.pop("low_cpu_mem_usage", None)
            pipe = pipeline_cls.from_pretrained(model_id, **kwargs)

        if lora_dir and Path(lora_dir).exists() and any(Path(lora_dir).iterdir()):
            lora_dir = Path(lora_dir)
            peft_dir = lora_dir / "transformer_peft"
            try:
                if peft_dir.exists() and hasattr(pipe.transformer, "load_adapter"):
                    pipe.transformer.load_adapter(str(peft_dir), adapter_name="bdd100k_road_lora")
                    pipe.transformer.set_adapter("bdd100k_road_lora")
                    print(f"Loaded PEFT transformer adapter from {peft_dir}")
                else:
                    pipe.load_lora_weights(str(lora_dir))
                    print(f"Loaded Diffusers LoRA from {lora_dir}")
            except Exception as exc:
                print("LoRA load skipped. The base model will be used for augmentation.")
                print(type(exc).__name__, exc)

        pipe.to(device)
        if hasattr(pipe, "enable_vae_slicing"):
            pipe.enable_vae_slicing()
        if hasattr(pipe, "enable_vae_tiling"):
            pipe.enable_vae_tiling()
        if hasattr(pipe, "enable_attention_slicing"):
            pipe.enable_attention_slicing()
    return pipe


def records_by_split_and_bucket(records):
    grouped = {}
    for record in records:
        grouped.setdefault(record.split, {}).setdefault(record.bucket, []).append(record)
    return grouped


def choose_records_for_bucket(bucket_records, target_count, rng):
    if not bucket_records:
        return []
    if len(bucket_records) >= target_count:
        return rng.sample(bucket_records, target_count)
    return [rng.choice(bucket_records) for _ in range(target_count)]


def choose_record_for_variant(bucket_records, variant, rng):
    target_weather = VARIANT_TARGET_WEATHER.get(variant)
    if target_weather:
        candidates = [record for record in bucket_records if target_weather not in (record.weather or "")]
        if candidates:
            return rng.choice(candidates)
    return rng.choice(bucket_records)


def generated_image_path(output_dir, record, variant, index):
    safe_variant = variant.replace("/", "_")
    file_name = f"{record.path.stem}_aug_{index:04d}_{safe_variant}.png"
    return Path(output_dir) / record.split / record.bucket / IMAGE_SUBDIR / file_name


def generate_variant_with_pipe(pipe, record, variant, output_path, seed, device=TRAIN_DEVICE, strength=None):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    source = resize_center_crop(load_source_image(record.path), resolution=RESOLUTION)
    prompt = build_generation_prompt(record, variant)
    negative_prompt = build_variant_negative_prompt(variant)
    strength = VARIANT_STRENGTHS.get(variant, AUGMENTATION_STRENGTH) if strength is None else strength
    guidance_scale = VARIANT_GUIDANCE_SCALES.get(variant, GUIDANCE_SCALE)
    if seed == SEED:
        print("Prompt sample:", prompt)
        print("Negative sample:", negative_prompt)
        print(f"Generation config: variant={variant}, strength={strength}, guidance_scale={guidance_scale}, steps={NUM_INFERENCE_STEPS}, resolution={RESOLUTION}")
    generator_device = device if str(device).startswith("cuda") else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=source,
        strength=strength,
        guidance_scale=guidance_scale,
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=generator,
    ).images[0]
    image.save(output_path)
    if str(device).startswith("cuda"):
        torch.cuda.empty_cache()
    return output_path


def write_manifest(rows, output_dir=OUTPUT_DIR):
    manifest_path = Path(output_dir) / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["split", "bucket", "source_weather", "source_timeofday", "source_scene", "variant", "source_path", "label_path", "output_path", "seed"]
    with manifest_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved manifest: {manifest_path}")
    return manifest_path


def build_augmentation_jobs(records, variants, target_per_bucket, target_splits):
    rng = random.Random(SEED)
    grouped = records_by_split_and_bucket(records)
    jobs = []
    job_index = 0
    for split in target_splits:
        for bucket in SCENE_BUCKETS:
            bucket_records = grouped.get(split, {}).get(bucket, [])
            if not bucket_records:
                print(f"SKIP {split}/{bucket}: no source images found")
                continue
            print(f"Queued {target_per_bucket} images for {split}/{bucket}")
            for bucket_index in range(1, target_per_bucket + 1):
                variant = variants[(bucket_index - 1) % len(variants)]
                record = choose_record_for_variant(bucket_records, variant, rng)
                output_path = generated_image_path(OUTPUT_DIR, record, variant, bucket_index)
                jobs.append({
                    "job_index": job_index,
                    "record": record,
                    "variant": variant,
                    "output_path": output_path,
                    "seed": SEED + job_index,
                })
                job_index += 1
    return jobs


def run_augmentation_jobs_on_device(device, jobs, total_jobs, backend):
    if not jobs:
        return [], []
    if str(device).startswith("cuda"):
        torch.cuda.set_device(torch.device(device).index or 0)
    print(f"[{device}] loading pipeline for {len(jobs)} jobs")
    pipe = build_img2img_pipeline(backend=backend, lora_dir=LORA_DIR, device=device)
    outputs = []
    rows = []
    for local_index, job in enumerate(jobs, 1):
        record = job["record"]
        saved = generate_variant_with_pipe(
            pipe=pipe,
            record=record,
            variant=job["variant"],
            output_path=job["output_path"],
            seed=job["seed"],
            device=device,
            strength=None,
        )
        outputs.append(saved)
        rows.append({
            "split": record.split,
            "bucket": record.bucket,
            "source_weather": record.weather or "",
            "source_timeofday": record.timeofday or "",
            "source_scene": record.scene or "",
            "variant": job["variant"],
            "source_path": str(record.path),
            "label_path": str(record.label_path) if record.label_path else "",
            "output_path": str(saved),
            "seed": job["seed"],
        })
        completed = job["job_index"] + 1
        if completed == 1 or completed % 25 == 0 or local_index == len(jobs):
            print(f"[{device}] [{completed}/{total_jobs}] saved {saved.name}")
    del pipe
    clear_cuda()
    return outputs, rows


def augment_dataset(records, variants=AUGMENTATION_VARIANTS, backend=MODEL_BACKEND, target_per_bucket=AUGMENTATIONS_PER_BUCKET, target_splits=TARGET_SPLITS):
    if not records:
        raise FileNotFoundError("No images found. Mount dataset folder and rerun scan_dataset().")
    devices = resolve_augmentation_devices()
    jobs = build_augmentation_jobs(records, variants, target_per_bucket, target_splits)
    if not jobs:
        print("No augmentation jobs were queued.")
        return []
    total_jobs = len(jobs)
    print(f"Using augmentation devices: {devices}")
    shards = [jobs[index::len(devices)] for index in range(len(devices))]
    all_outputs = []
    manifest_rows = []
    if len(devices) == 1:
        all_outputs, manifest_rows = run_augmentation_jobs_on_device(devices[0], shards[0], total_jobs, backend)
    else:
        with ThreadPoolExecutor(max_workers=len(devices)) as executor:
            futures = [
                executor.submit(run_augmentation_jobs_on_device, device, shard, total_jobs, backend)
                for device, shard in zip(devices, shards)
                if shard
            ]
            for future in as_completed(futures):
                outputs, rows = future.result()
                all_outputs.extend(outputs)
                manifest_rows.extend(rows)
    manifest_rows = sorted(manifest_rows, key=lambda row: row["seed"])
    all_outputs = [Path(row["output_path"]) for row in manifest_rows]
    write_manifest(manifest_rows, OUTPUT_DIR)
    print(f"Generated {len(all_outputs)} images in {OUTPUT_DIR}")
    return all_outputs


# Smoke test: 2 generated images for the train road-scene bucket only.
# generated_paths = augment_dataset(records, target_per_bucket=2, target_splits=["train"])

# Full requested batch: 200 generated images per split for train, val, and test.
# generated_paths = augment_dataset(records)
# generated_paths[:5]

## 11. Recommended First Run

1. Confirm `DATASET_ROOT` points to the Kaggle folder that contains `bdd100k/images/100k/{train,val,test}`.
2. Run dataset scanner and preview.
3. Smoke-test augmentation with `target_per_bucket=2` on `train`.
4. Then run LoRA training with small `MAX_TRAIN_STEPS` if you need a domain adapter.
5. Generate the requested batch with `augment_dataset(records)`, which creates 200 images for the road-scene bucket in each target split.

On Kaggle T4 x2, augmentation auto-detects both GPUs when `USE_ALL_GPUS_FOR_AUGMENTATION=True`. LoRA training remains on `TRAIN_DEVICE` by default.

In [ ]:
# Smoke-test augmentation after mounting dataset:
# generated_paths = augment_dataset(records, target_per_bucket=2, target_splits=["train"])

# Train over multiple dataset images after smoke test:
# trained_pipe = train_sd35_lora_on_records(records, max_steps=MAX_TRAIN_STEPS, rank=LORA_RANK)

# Full requested augmentation after training or with the base model:
# generated_paths = augment_dataset(records)